# Operations 07 Parsing and Retry (LangChain 2026)

## What This Lesson Is
Retry on invalid structured output with bounded attempt policy.

## Scientific Lens
- Concept: Parse-validate-retry loop
- Measure: Parse success within retry budget
- Validity Limit: Retry can hide deeper prompt/model problems.


## How It Works
1. Deterministically simulate parse failures.
2. Bound retries with explicit stop.
3. Run live model output through parser retries.


In [ ]:
print("Parsing-retry lesson preflight complete")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: real provider/tool path with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
import json
outputs=["{\"status\":\"ok\"}","invalid-json","{\"status\":\"ok\"}"]
parsed=None
for i,t in enumerate(outputs,1):
    try:
        parsed=json.loads(t)
        print("parsed attempt",i)
        break
    except Exception:
        print("fail attempt",i)
assert parsed and parsed.get("status")=="ok"


In [ ]:
# Live Demo
import os, json
try:
    from langchain_openai import ChatOpenAI
except Exception as exc:
    print(f"Skipping live parse run: dependency unavailable ({exc})")
else:
    if not os.getenv("OPENAI_API_KEY"):
        print("Skipping live parse run: OPENAI_API_KEY not set.")
    else:
        llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0.2, timeout=20)
        prompt = 'Return strict JSON with key status set to ok and no extra text.'
        for attempt in range(1, 4):
            out = llm.invoke(prompt)
            text = out.content if hasattr(out, 'content') else str(out)
            try:
                obj = json.loads(text)
                print('parsed', obj, 'attempt', attempt)
                break
            except Exception as exc:
                print('parse failed', attempt, exc)
        else:
            print('retry budget exhausted')


## Applied Labs
1. Add JSON schema validation post-parse.
2. Implement one automatic repair pass before retrying model.
3. Track parse-failure taxonomy across 30 runs.

## Validation Checklist
- Retry budget is bounded and explicit.
- Parse failures do not pass silently.
- Successful parse returns structured object.

## Further Reading
- [LangChain output parsers](https://python.langchain.com/docs/concepts/output_parsers/)
- [JSON Schema](https://json-schema.org/)
- [Robust parsing patterns](https://martinfowler.com/articles/replaceThrowWithNotification.html)
